# 📞 Aula 11 — A distribuição de Poisson na central de suporte de TI

Este notebook **não tem nota nem entrega**, e você **não precisa saber programar**. É só executar as células, na ordem, de cima para baixo, e ler os textos entre elas.

Neste notebook você faz parte do **time de dados de uma empresa de software**. A central de suporte (o *help desk*) recebe chamados ao longo do dia, e a gerência precisa entender esse fluxo para tomar decisões. A ferramenta é a da Aula 11: a **distribuição de Poisson**, que conta **quantas ocorrências** acontecem num **intervalo** (de tempo, de espaço...) quando conhecemos a **taxa média** $\lambda$.

> ⚠️ Os números usados aqui são **fictícios**, escolhidos só para ilustrar os modelos.

Cada seção segue sempre a mesma ordem:
1. **A situação e a pergunta** (o enunciado, como num exercício);
2. **A ferramenta da aula** que responde a pergunta;
3. **A resposta pela conta**;
4. **O que o código faz**, antes de cada célula;
5. **Como ler o resultado**.

> Notebook elaborado com auxílio de IA (Claude) e conferido pelo professor.

## Como executar

- Cada **célula de código** (as caixas cinza) tem um botão **▶** do lado esquerdo. Clique nele, ou clique na célula e aperte **Shift + Enter**.
- Execute **uma de cada vez, na ordem**: algumas células usam o resultado das anteriores.
- Na primeira vez, o Colab pode avisar que o notebook "não foi criado pelo Google". Clique em **Executar mesmo assim**.
- Se preferir, use o menu **Ambiente de execução → Executar tudo** e depois só leia, de cima para baixo.
- Detalhe: o Python escreve números com **ponto** no lugar da vírgula (`0.26` = 0,26).

## 0. Preparando as ferramentas

A célula abaixo carrega:

- **`math.exp` e `math.factorial`**: a exponencial $e^{x}$ e o fatorial $k!$, que aparecem na fórmula da Poisson;
- **`math.comb`**: combinações $\binom{n}{k}$, para a binomial da última seção;
- **`numpy`**: uma "calculadora" que faz contas com milhares de números de uma vez e sabe **sortear** números aleatórios;
- **`matplotlib`**: desenha **gráficos**.

Também escrevemos, uma única vez, a **fórmula da aula** como uma função do Python (uma "receita" com nome):

$$P(X=k) = \frac{e^{-\lambda}\,\lambda^{k}}{k!}, \qquad k = 0, 1, 2, \dots$$

Por fim, criamos o **sorteador** (`rng`) com a **semente 42**, que faz as simulações saírem sempre iguais.

**Lembrete:** **empírico** = o que sai da simulação; **teórico** = o que a fórmula da aula diz.

In [ ]:
from math import exp, factorial, comb   # e^x, k! e combinações
import numpy as np                       # calculadora de números e sorteios
import matplotlib.pyplot as plt          # gráficos


def poisson(k, lam):
    # P(X = k) na Poisson com taxa média lam
    return exp(-lam) * lam**k / factorial(k)


def binom(k, n, p):
    # P(X = k) na binomial: n ensaios independentes, probabilidade de sucesso p
    return comb(n, k) * p**k * (1 - p)**(n - k)


rng = np.random.default_rng(42)          # sorteador, semente 42
print("Tudo pronto!")

## 1. Quantos chamados chegam numa hora?

### A situação
Pelo histórico, a central de suporte recebe em média **4 chamados por hora**, e os chamados chegam de forma independente uns dos outros (um cliente não combina com o outro a hora de abrir um chamado).

### A pergunta
Seja $X$ = número de chamados que chegam em **uma hora**.
1. Qual a chance de passar uma hora **sem nenhum** chamado?
2. A equipe considera "sobrecarga" uma hora com **8 ou mais** chamados. Qual a chance de sobrecarga?

### A ferramenta da aula
Contamos ocorrências num intervalo de tempo, com taxa média conhecida: $X \sim \text{Pois}(\lambda = 4)$.

### A resposta pela conta
- $P(X = 0) = \dfrac{e^{-4}\,4^0}{0!} = e^{-4} \approx 0{,}0183$;
- $P(X \ge 8) = 1 - \big[P(X=0) + P(X=1) + \dots + P(X=7)\big] \approx 1 - 0{,}9489 = 0{,}0511$.

### O que o código faz
1. calcula as duas respostas com a função `poisson`;
2. **simula 100.000 horas de expediente** (`rng.poisson` sorteia quantos chamados chegam em cada hora) e conta em quantas delas não houve chamado e em quantas houve sobrecarga;
3. compara também a **média** e a **variância** dos chamados simulados com o que a aula diz: na Poisson, $E(X) = \text{Var}(X) = \lambda$.

In [ ]:
lam = 4                                                    # 4 chamados por hora, em média

p_zero = poisson(0, lam)
p_sobrecarga = 1 - sum(poisson(k, lam) for k in range(0, 8))   # 1 - P(X <= 7)

horas = rng.poisson(lam, size=100_000)                     # 100.000 horas simuladas

print(f"P(X = 0)   teórico: {p_zero:.4f}   empírico: {(horas == 0).mean():.4f}")
print(f"P(X >= 8)  teórico: {p_sobrecarga:.4f}   empírico: {(horas >= 8).mean():.4f}")
print(f"Média      teórico: {lam:.2f}     empírico: {horas.mean():.2f}")
print(f"Variância  teórico: {lam:.2f}     empírico: {horas.var():.2f}")

**Como ler:**
- Uma hora **sem nenhum chamado** é rara: cerca de **1,8%** das horas (mais ou menos 1 hora a cada 55).
- **Sobrecarga** (8 ou mais chamados) acontece em cerca de **5%** das horas — numa jornada de 8 horas por dia, isso dá uma hora de sobrecarga a cada 2 ou 3 dias.
- A média e a variância simuladas ficam as duas perto de **4**: é a "assinatura" da Poisson, em que a esperança e a variância são iguais a $\lambda$.

Para ver a distribuição inteira, o código abaixo calcula $P(X=k)$ de 0 a 12 chamados e desenha um **gráfico de barras**, com as horas de sobrecarga em vermelho.

In [ ]:
valores = range(0, 13)
probs = [poisson(k, lam) for k in valores]
cores = ["firebrick" if k >= 8 else "steelblue" for k in valores]   # vermelho = sobrecarga

for k, pk in zip(valores, probs):
    print(f"P(X = {k:2d}) = {pk:.4f}")

plt.bar(valores, probs, color=cores, edgecolor="black")
plt.xlabel("Chamados em uma hora")
plt.ylabel("Probabilidade")
plt.title("Poisson com λ = 4 — em vermelho, as horas de sobrecarga")
plt.show()

**Como ler:** os valores mais prováveis são **3 e 4 chamados** (cerca de 19,5% cada), bem em torno da média $\lambda = 4$. A distribuição tem uma "cauda" para a direita: valores grandes como 10 ou 11 chamados são possíveis, mas muito raros.

## 2. Mudando o tamanho do intervalo

### A situação e a pergunta
Um atendente quer sair para um café e pergunta ao time de dados: **qual a chance de não chegar nenhum chamado nos próximos 15 minutos? E nos próximos 30?**

### A ferramenta da aula
A taxa $\lambda$ é sempre **por intervalo**. Se chegam 4 chamados por hora, então chegam, em média:
- $4 \times \frac{15}{60} = 1$ chamado em 15 minutos;
- $4 \times \frac{30}{60} = 2$ chamados em 30 minutos.

Em $t$ minutos, a taxa é $\lambda_t = 4 \cdot \frac{t}{60}$, e a chance de nenhum chamado é $P(X=0) = e^{-\lambda_t}$ — é o mesmo raciocínio do exemplo da central telefônica da aula.

### A resposta pela conta
- 15 minutos: $e^{-1} \approx 0{,}3679$;
- 30 minutos: $e^{-2} \approx 0{,}1353$.

### O que o código faz
1. calcula a chance de "nenhum chamado" para 15 e 30 minutos;
2. repete a conta para todas as pausas de 1 a 60 minutos e desenha a curva "duração da pausa × chance de não perder nenhum chamado".

In [ ]:
for t in (15, 30):
    lam_t = 4 * t / 60                                     # taxa ajustada para t minutos
    print(f"{t} min: λ = {lam_t:.0f}   P(nenhum chamado) = {poisson(0, lam_t):.4f}")

minutos = np.arange(1, 61)                                 # pausas de 1 a 60 minutos
chance = np.exp(-4 * minutos / 60)                         # e^(-λt) para cada pausa

plt.plot(minutos, chance, color="steelblue")
plt.axhline(0.5, color="firebrick", linestyle="--", label="50% de chance")
plt.xlabel("Duração da pausa (minutos)")
plt.ylabel("Chance de não chegar nenhum chamado")
plt.title("Quanto maior a pausa, menor a chance de não perder nenhum chamado")
plt.legend()
plt.show()

**Como ler:** numa pausa de **15 minutos**, há cerca de **37%** de chance de não chegar nenhum chamado; em **30 minutos**, só **14%**. A curva cai rápido: a chance fica abaixo de 50% já com uma pausa de pouco mais de 10 minutos.

**Conclusão para a gerência:** com 4 chamados por hora, um atendente sozinho quase sempre vai perder algum chamado durante uma pausa. Pausas precisam de alguém cobrindo.

## 3. Quantos chamados a equipe precisa dar conta por hora?

### A situação e a pergunta
A gerência quer dimensionar a equipe para dar conta de todos os chamados em **pelo menos 95% das horas**. **Qual o menor número de chamados por hora, $c$, que a equipe precisa ser capaz de atender?**

### A ferramenta da aula
Ainda $X \sim \text{Pois}(4)$. Queremos o **menor** $c$ tal que
$$P(X \le c) = P(X=0) + P(X=1) + \dots + P(X=c) \ge 0{,}95 .$$

### O que o código faz
Vai somando as probabilidades $P(X=0), P(X=1), \dots$ (a **probabilidade acumulada**) e para no primeiro $c$ em que a soma passa de 95%.

In [ ]:
acumulada = 0
for c in range(0, 20):
    acumulada += poisson(c, lam)                           # P(X <= c)
    print(f"capacidade de {c:2d} chamados/hora: dá conta em {acumulada:.1%} das horas")
    if acumulada >= 0.95:
        break

**Como ler:** com capacidade para **7** chamados por hora, a equipe dá conta em 94,9% das horas — por um triz abaixo da meta. Com capacidade para **8**, passa para **97,9%**.

**Recomendação do time de dados:** dimensionar a equipe para **8 chamados por hora**, o dobro da média. Repare que dimensionar só pela média (4 por hora) seria um erro: a equipe ficaria sobrecarregada em cerca de 37% das horas (é a chance de $X \ge 5$).

## 4. A Poisson como atalho da binomial: detecção de fraude

### A situação
O sistema de pagamentos processou **10.000 transações** hoje. Cada transação tem probabilidade **0,0003** de ser fraude, independentemente das outras.

### A pergunta
Seja $X$ = número de fraudes do dia. Qual a chance de haver **exatamente 5** fraudes? E **pelo menos 5**?

### A ferramenta da aula
O modelo exato é a **binomial** $B(10.000;\ 0{,}0003)$: são $n$ ensaios independentes com a mesma $p$. Mas $n$ é muito grande e $p$ muito pequeno — exatamente o caso em que a aula mostrou que a **Poisson aproxima a binomial**, com
$$\lambda = n\,p = 10.000 \times 0{,}0003 = 3 .$$

### O que o código faz
Calcula as mesmas probabilidades pelas duas fórmulas, lado a lado: a binomial (exata, conta trabalhosa) e a Poisson com $\lambda = 3$ (aproximação, conta simples).

In [ ]:
n_trans, p_fraude = 10_000, 0.0003
lam_fraude = n_trans * p_fraude                            # λ = n·p = 3

print(" k    binomial    Poisson")
for k in range(0, 8):
    print(f" {k}    {binom(k, n_trans, p_fraude):.5f}    {poisson(k, lam_fraude):.5f}")

pb = 1 - sum(binom(k, n_trans, p_fraude) for k in range(0, 5))
pp = 1 - sum(poisson(k, lam_fraude) for k in range(0, 5))
print(f"\nP(X >= 5)   binomial: {pb:.5f}   Poisson: {pp:.5f}")

**Como ler:** as duas colunas são praticamente iguais — por exemplo, $P(X=5)$ dá 0,10082 pelas duas, e $P(X \ge 5) \approx 0{,}1847$ pelas duas. A chance de um dia com 5 ou mais fraudes é de cerca de **18%**.

**Por que isso importa:** em ciência de dados é comum contar eventos **raros** (fraudes, falhas, erros) em bases **enormes**. Nesses casos, ninguém usa a binomial com $n = 10.000$: usa-se a Poisson com $\lambda = n\,p$, que dá o mesmo resultado com uma conta muito mais simples.

## Para fechar

Neste notebook, o time de dados usou a **distribuição de Poisson** da Aula 11 para:

- calcular a chance de horas sem chamados e de horas de sobrecarga ($\lambda = 4$ por hora);
- **ajustar a taxa ao tamanho do intervalo** (15 minutos → $\lambda = 1$; 30 minutos → $\lambda = 2$);
- **dimensionar a equipe** pela probabilidade acumulada;
- **aproximar a binomial** quando $n$ é grande e $p$ é pequeno ($\lambda = n\,p$).

Na **Aula 12** vêm os primeiros **modelos contínuos**. Um deles, a **exponencial**, mede o **tempo entre** dois chamados — a outra face da mesma história da Poisson.